In [5]:
import pandas as pd
df= pd.read_csv("../DATA/spam_processed.csv")
import pickle
import re
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.metrics import   confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
with open("../MODELS/tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)
with open("../MODELS/spam_classifier_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("../MODELS/tfidf2.pkl", "rb") as f:
    tfidf2= pickle.load(f)
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df["c_e"], df["label"], test_size=0.2, random_state=42,stratify=df["label"])
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
with open("../MODELS/logistic_regression_model.pkl", "rb") as f:
    lr_model= pickle.load(f)
with open("../MODELS/svm_model.pkl", "rb") as f:
    svm_model= pickle.load(f)
from collections import Counter
from sklearn.metrics import classification_report
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
def trans_text(text):
    text=text.lower()
    text=re.sub(r'[^a-zA-Z0-9 ]', '', text)
    text=text.split()
    text1=[]
    for word in text:
        if word not in stop_words:
            word=ps.stem(word)
            text1.append(word)
    return " ".join(text1)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words1 = set(ENGLISH_STOP_WORDS)
x_train_tf=tfidf.fit_transform(x_train)
x_test_tf=tfidf.transform(x_test)

[nltk_data] Downloading package stopwords to C:\Users\Mr
[nltk_data]     X\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
svm_model = LinearSVC()
svm_model.fit(x_train_tf, y_train)
svm_y_pred=svm_model.predict(x_test_tf)
with open("../MODELS/svm_model.pkl", "wb") as f:
    pickle.dump(svm_model, f)
print("SVM Model Evaluation:")
print(classification_report(y_test, svm_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_y_pred))

In [ ]:
results=pd.DataFrame({"actual":y_test, "predicted":svm_y_pred})
wrong_results=results[results["actual"]!=results["predicted"]]
wrong_indices=wrong_results.index
print(wrong_indices)
print(df.loc[wrong_indices,["email","c_e","label"]].head(20))


In [ ]:

tfidf2=TfidfVectorizer(ngram_range=(1, 2))
with open("../MODELS/tfidf2.pkl", "wb") as f:
    pickle.dump(tfidf2, f)
x_train_tf2 = tfidf2.fit_transform(x_train)
x_test_tf2 = tfidf2.transform(x_test)
svm_model.fit(x_train_tf2, y_train)
svm_y_pred2 = svm_model.predict(x_test_tf2)
print(confusion_matrix(y_test, svm_y_pred2))


In [ ]:
results=pd.DataFrame({"actual":y_test, "predicted":svm_y_pred2})
wrong_results=results[results["actual"]!=results["predicted"]]
wrong_indices=wrong_results.index
print(wrong_indices)
print(df.loc[wrong_indices,["email","c_e","label"]].head(10))

In [15]:
def trans_text1(text):
    text=text.lower()
    text=re.sub(r'[^a-zA-Z0-9?!. ]', '', text)
    text=text.split()
    text1=[]
    for word in text:
        if word not in stop_words:
            
            text1.append(word)
    return " ".join(text1)
df["c_e1"]=df["email"].apply(trans_text1)
df.to_csv("../DATA/spam_processed.csv")
x_train1, x_test1, y_train, y_test = train_test_split(df["c_e1"], df["label"], test_size=0.2, random_state=42,stratify=df["label"])
x_train_tf1=tfidf2.fit_transform(x_train1)
x_test_tf1=tfidf2.transform(x_test1)
svm_model.fit(x_train_tf1, y_train)
svm_y_pred1=svm_model.predict(x_test_tf1)
print(confusion_matrix(y_test, svm_y_pred1))
print(classification_report(y_test, svm_y_pred1))
with open("../MODELS/tfidf2.pkl", "wb") as f:
    pickle.dump(tfidf2, f)
with open("../MODELS/svm_model.pkl", "wb") as f:
    pickle.dump(svm_model, f)
    print(df["c_e1"])

[[490   0]
 [  8  77]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       490
           1       1.00      0.91      0.95        85

    accuracy                           0.99       575
   macro avg       0.99      0.95      0.97       575
weighted avg       0.99      0.99      0.99       575

0       date wed number aug number number number numbe...
1       martin posted tassos papadopoulos greek sculpt...
2       man threatens explosion moscow thursday august...
3       klez virus die already prolific virus ever kle...
4       adding cream spaghetti carbonara effect pasta ...
                              ...                        
2866    abc good morning america ranks number christma...
2867    hyperlink hyperlink hyperlink let mortgage len...
2868    thank shopping us gifts occasions free gift nu...
2869    famous ebay marketing e course learn sell comp...
2870    hello chinese traditional number number f r v ...
Name: c_e1, 

In [16]:
new_email =input("entrer votre email:")
def predict_email(new_email):
    clean_email= trans_text1(new_email)
    new_email_tf=tfidf2.transform([clean_email])
    new_email_pred = svm_model.predict(new_email_tf)
    if new_email_pred == 1:
        return ("The new email is classified as: Spam")
    else:
        return ("The new email is classified as: Ham")
predict_email(new_email)

'The new email is classified as: Spam'

In [ ]:
prize winner! click here to claim your free gift card now!

In [ ]:
print(df.columns)
x=df[["c_e1","word_count","character_count"]]
print(x)